# Question 3: Testing and Validation

Simple but effective testing for a 5-hour assignment.

## Testing Strategy

We test the essential functionality:
1. ✅ Individual tool functionality
2. ✅ Agent orchestration
3. ✅ Error handling
4. ✅ Complete analysis flow

## Running Tests

Our tests are in `tests/test_agent.py`. Let's run them:

In [ ]:
# Run tests
import subprocess
result = subprocess.run(['pytest', '../tests/test_agent.py', '-v'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

## Manual Testing: Complete Flow

Let's manually test the complete analysis flow:

In [ ]:
import sys
sys.path.append('..')

from src.agent.orchestrator import MarketAnalysisAgent
from src.tools.product_collector import ProductCollectorTool
from src.tools.sentiment_analyzer import SentimentAnalyzerTool
from src.tools.report_generator import ReportGeneratorTool
from src.utils.models import AnalysisRequest

# Setup agent
agent = MarketAnalysisAgent()
agent.register_tool(ProductCollectorTool(use_mock_data=True))
agent.register_tool(SentimentAnalyzerTool(use_llm=False))
agent.register_tool(ReportGeneratorTool(use_llm=False))

# Test 1: iPhone Analysis
print("TEST 1: iPhone 15 Pro Analysis")
print("="*60)
request = AnalysisRequest(
    product_query="iPhone 15 Pro",
    include_competitors=True,
    include_sentiment=True
)
result = agent.analyze(request)

print(f"\n✅ Product: {result.product_data.name if result.product_data else 'N/A'}")
print(f"✅ Price: ${result.product_data.price if result.product_data else 0}")
print(f"✅ Sentiment: {result.sentiment.overall_sentiment if result.sentiment else 'N/A'}")
print(f"✅ Competitors: {len(result.competitors)}")
print(f"✅ Recommendations: {len(result.recommendations)}")

## Error Handling Tests

In [ ]:
# Test 2: Unknown Product (Error Handling)
print("\nTEST 2: Error Handling - Unknown Product")
print("="*60)

request2 = AnalysisRequest(
    product_query="Completely Unknown Product XYZ123",
    include_competitors=False,
    include_sentiment=False
)
result2 = agent.analyze(request2)

print(f"\n✅ Still completes: {result2 is not None}")
print(f"✅ Has product data: {result2.product_data is not None}")
print(f"✅ Fallback product: {result2.product_data.name if result2.product_data else 'N/A'}")
print("\n✔️ System handles unknowns gracefully!")

## Output Validation

In [ ]:
# Test 3: Output Structure Validation
print("\nTEST 3: Output Validation")
print("="*60)

# Check all required fields
checks = [
    (result.product_data is not None, "Product data present"),
    (result.product_data.name != "", "Product has name"),
    (result.product_data.price > 0, "Product has valid price"),
    (result.sentiment is not None, "Sentiment data present"),
    (result.sentiment.overall_sentiment in ["positive", "negative", "neutral"], "Valid sentiment"),
    (len(result.recommendations) > 0, "Recommendations generated"),
    (len(result.competitors) > 0, "Competitors found"),
]

passed = 0
for check, description in checks:
    status = "✅" if check else "❌"
    print(f"{status} {description}")
    if check:
        passed += 1

print(f"\n📊 Passed: {passed}/{len(checks)} checks")

## Generate Example Reports

In [ ]:
# Generate full markdown report
if result.metadata.get("report") and "markdown_report" in result.metadata["report"]:
    print("\n📄 GENERATED REPORT PREVIEW:")
    print("="*60)
    report = result.metadata["report"]["markdown_report"]
    # Show first 30 lines
    lines = report.split('\n')[:30]
    print('\n'.join(lines))
    print("\n... (full report in metadata)")
    
    # Save to file
    import os
    os.makedirs('../reports', exist_ok=True)
    with open('../reports/test_report.md', 'w') as f:
        f.write(report)
    print("\n✅ Full report saved to: reports/test_report.md")

## Summary

### Test Coverage

✅ **Unit Tests** (10 tests):
- Product Collector: 4 tests
- Sentiment Analyzer: 4 tests  
- Report Generator: 2 tests

✅ **Integration Tests** (4 tests):
- Agent initialization
- Tool registration
- Complete analysis flow
- Partial analysis flow

✅ **Error Handling** (3 tests):
- Empty inputs
- Missing tools
- Invalid data

### Testing Approach

**Simple but effective** - appropriate for a 5-hour assignment:
- Focus on critical paths
- Test happy path and key error cases
- Validate output structure
- Use mock data (no external dependencies)

### Running Tests

```bash
# Run all tests
pytest tests/test_agent.py -v

# Run with coverage
pytest tests/test_agent.py --cov=src --cov-report=html

# In Docker
docker-compose run agent-interactive
```